# 04 DPO：Verifier-guided Offline Preference Optimization

本节实现离线 DPO 主线：`SFT Policy → Rollout → Verifier → Chosen / Rejected → DPO`。偏好标签由确定性答案验证器产生，DPO Loss、Response Log Probability 和 Reference Policy 都在 Notebook 中显式实现。

当前 DGX Spark 阶段不加载模型、不启动 vLLM、不生成真实 Rollout。CPU Smoke Test 使用带有 `fixture=True` 标记的极小数值样本，只验证偏好构造和梯度方向，不把 Fixture 当作模型实验结果。

## 学习目标

1. 记录多候选 Rollout 所需的 Token IDs、Response Mask、策略版本和截断状态；
2. 用独立 Verifier 选择最短正确 Chosen 与最短错误 Rejected；
3. 区分 DPO-A、Hard Negative 和 External Oracle 三类偏好构造；
4. 实现 Response Log Probability、Reference Log Probability 和标准 Sigmoid DPO Loss；
5. 验证 Reference Policy 不更新，且 DPO 的概率差异来自当前 Policy 更新。

## 1. 实验边界与控制变量

DPO 和后续 PPO/GRPO 都从同一个最佳 SFT Checkpoint 平行出发。Gravity 主要作为能力保持控制组；DPO 主线优先使用 Equation Numeric 和 Bit Manipulation。所有偏好对都必须记录 Verifier 版本、数据来源、策略版本和采样参数。

标准 DPO 目标：

\[
L_{DPO}=-\log\sigma\left(\beta\left[(\log\pi_\theta(y_w|x)-\log\pi_{ref}(y_w|x))- (\log\pi_\theta(y_l|x)-\log\pi_{ref}(y_l|x))\right]\right)
\]

Reference Policy 固定为初始化的最佳 SFT 策略：`pi_ref = pi_SFT`。本 Notebook 不依赖 TRL 的 `DPOTrainer`。

In [ ]:
from __future__ import annotations

import json
import math
import os
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd()
SYNTHETIC_ACCEPTED_PATH = PROJECT_ROOT / "data" / "processed" / "synthetic" / "synthetic_accepted.jsonl"
RESULT_DIR = PROJECT_ROOT / "data" / "processed" / "dpo"
CHECKPOINT_ROOT = PROJECT_ROOT / "data" / "processed" / "checkpoints"
SFT_MANIFEST_PATH = CHECKPOINT_ROOT / "sft_manifest.json"
TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "splits" / "train.csv"
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
BASE_MODEL_PATH = os.environ.get("QWEN3_MODEL_PATH", MODEL_ID)
SEED = 42
RUN_ROLLOUT = os.environ.get("RUN_ROLLOUT", "0") == "1"
RUN_DPO = os.environ.get("RUN_DPO", "0") == "1"
RUN_MODE = os.environ.get("RUN_MODE", "smoke")
SMOKE_PROMPTS_PER_TASK = 2
NUM_CANDIDATES_PER_PROMPT = 4
MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.7
TOP_P = 0.9
DPO_BETA = 0.1
LEARNING_RATE = 1e-5
TRAINING_PROFILE = "official-pt-rl-cuda"

np.random.seed(SEED)
torch.manual_seed(SEED)

if not SYNTHETIC_ACCEPTED_PATH.exists():
    raise FileNotFoundError("缺少第一个 Notebook 的 screened synthetic accepted 数据")
accepted_records = [json.loads(line) for line in SYNTHETIC_ACCEPTED_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
accepted_records = [row for row in accepted_records if row.get("task_type") in {"equation_numeric", "bit_manipulation"}]
train_df = pd.read_csv(TRAIN_PATH, dtype=str, keep_default_na=False) if TRAIN_PATH.exists() else pd.DataFrame()

def select_smoke_prompts(per_task: int = SMOKE_PROMPTS_PER_TASK) -> list[dict[str, object]]:
    if train_df.empty:
        # Fallback is still Train-side screened data, never Validation/Holdout.
        source = accepted_records
        return [dict(row, id=row.get("candidate_id"), source_split="train_synthetic") for task in ("equation_numeric", "bit_manipulation") for row in [x for x in source if x.get("task_type") == task][:per_task]]
    selected = []
    for task in ("equation_numeric", "bit_manipulation"):
        for row in train_df[train_df["task_type"] == task].sample(n=min(per_task, int((train_df["task_type"] == task).sum())), random_state=SEED + (1 if task == "equation_numeric" else 2)).itertuples(index=False):
            selected.append({"id": str(row.id), "task_type": str(row.task_type), "prompt": str(row.prompt), "answer": str(row.answer), "source_split": "train"})
    return selected

print({"model_id": MODEL_ID, "accepted_rl_prompts": len(accepted_records), "run_rollout": RUN_ROLLOUT, "run_dpo": RUN_DPO})

## 2. 确定性 Verifier 与 Rollout Schema

答案正确不等于推理轨迹正确；本节先以最终答案作为 DPO-A 主线标签，并把 `trace_verified`、`source` 等字段保留下来，为 Process Preference 扩展留出接口。数值任务使用第一个 Notebook 的官方容差，Bit 使用 8-bit 精确匹配。

In [ ]:
OFFICIAL_VERIFIER_VERSION = "official_nvidia_metric_v2"


def normalize_answer_text(value: object) -> str:
    text = str(value).strip()
    return __import__("re").sub(r"^```(?:text|json)?\s*|\s*```$", "", text, flags=__import__("re").I).strip()


def extract_final_answer(text: str, task_type: str) -> tuple[str | None, str]:
    """Notebook 01-compatible extraction: explicit markers first, fallbacks last."""
    import re
    text = str(text)
    boxed = re.findall(r"\\boxed\{([^{}]+)\}", text)
    if boxed:
        return normalize_answer_text(boxed[-1]), "boxed"
    tagged = re.findall(r"<(?:final|answer)>\s*(.*?)\s*</(?:final|answer)>", text, flags=re.I | re.S)
    if tagged:
        return normalize_answer_text(tagged[-1]), "tagged"
    if task_type == "bit_manipulation":
        matches = re.findall(r"(?<![01])[01]{8}(?![01])", text)
        if matches:
            return matches[-1], "binary_fallback"
    if task_type in {"gravity", "equation_numeric"}:
        matches = re.findall(r"(?<![A-Za-z])[-+]?\d+(?:\.\d+)?", text)
        if matches:
            return matches[-1], "numeric_fallback"
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return (normalize_answer_text(lines[-1]), "last_nonempty_line") if lines else (None, "empty")


def official_numeric_match(predicted: object, expected: object) -> bool:
    try:
        predicted_value, expected_value = float(normalize_answer_text(predicted)), float(normalize_answer_text(expected))
    except (TypeError, ValueError):
        return False
    return math.isfinite(predicted_value) and math.isfinite(expected_value) and math.isclose(predicted_value, expected_value, rel_tol=1e-2, abs_tol=1e-5)


def verify_answer(task_type: str, predicted: object, expected: object) -> tuple[bool, str]:
    if predicted is None:
        return False, "not_extracted"
    predicted_text, expected_text = normalize_answer_text(predicted), normalize_answer_text(expected)
    if task_type == "bit_manipulation":
        p, e = predicted_text.removeprefix("0b"), expected_text.removeprefix("0b")
        if not __import__("re").fullmatch(r"[01]{8}", p) or not __import__("re").fullmatch(r"[01]{8}", e):
            return False, "invalid_8bit_format"
        return p == e, "exact_8bit"
    if task_type in {"gravity", "equation_numeric"}:
        if not __import__("re").fullmatch(r"[-+]?\d+(?:\.\d+)?", predicted_text):
            return False, "invalid_numeric_format"
        return official_numeric_match(predicted_text, expected_text), "official_math_isclose"
    return predicted_text == expected_text, "exact_text"


def extract_final_answer_for_rollout(text: str, task_type: str) -> str | None:
    return extract_final_answer(text, task_type)[0]


def verify_completion(task_type: str, completion: str, expected: str, truncated: bool = False) -> tuple[bool, str]:
    if truncated:
        return False, "truncated_output"
    predicted, _ = extract_final_answer(completion, task_type)
    return verify_answer(task_type, predicted, expected)


@dataclass
class RolloutRecord:
    sample_id: str
    task_type: str
    prompt: str
    completion: str
    expected_answer: str
    completion_token_ids: list[int]
    prompt_token_ids: list[int]
    response_mask: list[int]
    finish_reason: str
    truncated: bool
    policy_version: str
    verifier_version: str
    temperature: float
    top_p: float
    source: str
    fixture: bool = False
    trace_verified: bool = False
    correct: bool = False
    verification_method: str = ""


def record_to_dict(record: RolloutRecord) -> dict[str, object]:
    return {
        "sample_id": record.sample_id, "task_type": record.task_type, "prompt": record.prompt,
        "completion": record.completion, "expected_answer": record.expected_answer,
        "completion_token_ids": record.completion_token_ids, "prompt_token_ids": record.prompt_token_ids,
        "response_mask": record.response_mask, "finish_reason": record.finish_reason,
        "truncated": record.truncated, "policy_version": record.policy_version,
        "verifier_version": record.verifier_version, "temperature": record.temperature, "top_p": record.top_p,
        "source": record.source, "fixture": record.fixture, "trace_verified": record.trace_verified,
        "correct": record.correct, "verification_method": record.verification_method,
    }


## 3. CPU Fixture 和偏好对构造

CPU Fixture 为每个测试 Prompt 放入一条“正确格式答案”和一条“错误答案”，仅用于让选择逻辑可运行；它不代表模型采样，也不产生实验 Accuracy。真实运行时，`run_vllm_rollout` 必须替换 Fixture，并保存实际 Token IDs 和策略版本。

In [ ]:
def build_cpu_fixture_rollouts(samples: list[dict[str, object]], candidates: int = 4) -> list[dict[str, object]]:
    records = []
    for sample in samples[: SMOKE_PROMPTS_PER_TASK * 2]:
        task = str(sample["task_type"])
        answer = str(sample["answer"])
        good = f"Reasoning verified from all public examples. Final answer: \\boxed{{{answer}}}"
        wrong = "The examples were not fully checked. Final answer: \\boxed{0}"
        completions = [good, wrong] + [wrong + f" candidate {i}" for i in range(max(0, candidates - 2))]
        for index, completion in enumerate(completions):
            correct, method = verify_completion(task, completion, answer)
            records.append(record_to_dict(RolloutRecord(
                sample_id=str(sample.get("candidate_id", sample.get("id"))),
                task_type=task,
                prompt=str(sample["prompt"]),
                completion=completion,
                expected_answer=answer,
                completion_token_ids=list(range(1, len(completion.split()) + 1)),
                prompt_token_ids=list(range(100, 100 + len(str(sample["prompt"]).split()))),
                response_mask=[1] * len(completion.split()),
                finish_reason="stop",
                truncated=False,
                policy_version="cpu_fixture",
                verifier_version=OFFICIAL_VERIFIER_VERSION,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                source="cpu_fixture_only",
                fixture=True,
                trace_verified=True,
                correct=correct,
                verification_method=method,
            )))
    return records


def _completion_length(record: dict[str, object]) -> int:
    return len(record.get("completion_token_ids", []))


def construct_preference_pairs(rollouts: list[dict[str, object]], mode: str = "dpo_a") -> list[dict[str, object]]:
    """Construct Chosen/Rejected pairs after independent verification."""
    if mode not in {"dpo_a", "hard_negative", "external_oracle"}:
        raise ValueError(mode)
    grouped = defaultdict(list)
    for record in rollouts:
        grouped[(record["sample_id"], record["task_type"])].append(record)
    pairs = []
    for (sample_id, task_type), group in grouped.items():
        correct = [row for row in group if row.get("correct") and not row.get("truncated")]
        incorrect = [row for row in group if not row.get("correct") and not row.get("truncated")]
        if mode == "external_oracle":
            oracle = next((row for row in group if row.get("source") == "deterministic_solver_oracle" and row.get("correct") and not row.get("truncated")), None)
            if oracle is not None:
                correct = [oracle]
        if not correct or not incorrect:
            continue
        chosen = min(correct, key=_completion_length)
        rejected = min(incorrect, key=_completion_length)
        pairs.append({
            "sample_id": sample_id,
            "task_type": task_type,
            "prompt": chosen["prompt"],
            "chosen": chosen["completion"],
            "rejected": rejected["completion"],
            "chosen_token_ids": chosen["completion_token_ids"],
            "rejected_token_ids": rejected["completion_token_ids"],
            "chosen_correct": True,
            "rejected_correct": False,
            "chosen_trace_verified": bool(chosen.get("trace_verified")),
            "chosen_truncated": bool(chosen.get("truncated")),
            "rejected_truncated": bool(rejected.get("truncated")),
            "preference_mode": mode,
            "verifier_version": chosen["verifier_version"],
            "source": "rollout_verifier",
            "fixture": bool(chosen.get("fixture", False)),
        })
    return pairs


def audit_preference_pairs(pairs: list[dict[str, object]]) -> dict[str, object]:
    reasons = Counter()
    valid = []
    for pair in pairs:
        checks = {
            "chosen_correct": pair.get("chosen_correct") is True,
            "rejected_incorrect": pair.get("rejected_correct") is False,
            "different_text": pair.get("chosen") != pair.get("rejected"),
            "same_prompt": bool(pair.get("prompt")),
            "not_truncated_chosen": not pair.get("chosen_truncated", False) and not pair.get("rejected_truncated", False),
        }
        if all(checks.values()):
            valid.append(pair)
        else:
            reasons.update(name for name, ok in checks.items() if not ok)
    return {"input_pairs": len(pairs), "valid_pairs": len(valid), "rejection_reasons": dict(reasons), "fixture_pairs": int(sum(pair.get("fixture", False) for pair in valid))}


fixture_rollouts = build_cpu_fixture_rollouts(select_smoke_prompts(), NUM_CANDIDATES_PER_PROMPT)
preference_pairs = construct_preference_pairs(fixture_rollouts, "dpo_a")
print({"fixture_rollouts": len(fixture_rollouts), "preference_pairs": len(preference_pairs), "audit": audit_preference_pairs(preference_pairs)})

## 4. RL Readiness Audit（DPO 前置诊断）

DPO 不要求 GRPO 式组内方差，但同一批 Rollout 仍应报告 All-correct / Mixed / All-wrong 覆盖，避免把极少数偏好对误认为充分训练信号。

In [ ]:
def rl_readiness_audit(rollouts: list[dict[str, object]], group_size: int = NUM_CANDIDATES_PER_PROMPT) -> dict[str, object]:
    grouped = defaultdict(list)
    for row in rollouts:
        grouped[(row["sample_id"], row["task_type"])].append(int(bool(row.get("correct"))))
    categories = Counter()
    variances = []
    for rewards in grouped.values():
        rewards = rewards[:group_size]
        correct_count = sum(rewards)
        if correct_count == len(rewards): categories["all_correct"] += 1
        elif correct_count == 0: categories["all_wrong"] += 1
        else: categories["mixed"] += 1
        variances.append(float(np.var(rewards)))
    total = sum(categories.values())
    return {
        "groups": total,
        "categories": dict(categories),
        "all_correct_ratio": categories["all_correct"] / total if total else 0.0,
        "mixed_ratio": categories["mixed"] / total if total else 0.0,
        "all_wrong_ratio": categories["all_wrong"] / total if total else 0.0,
        "mean_reward_variance": float(np.mean(variances)) if variances else 0.0,
        "fixture_only": bool(rollouts) and all(row.get("fixture", False) for row in rollouts),
    }


print(json.dumps({"rl_readiness": rl_readiness_audit(fixture_rollouts)}, ensure_ascii=False, indent=2))

## 5. Response Log Probability 与 DPO Loss

只对 Completion Token 计算序列 Log Probability；Prompt Token、Padding 和截断后的无效位置都不参与。DPO 的 Chosen / Rejected 使用同一 Prompt，Reference Log Probability 在训练期间 `no_grad` 且参数冻结。

In [ ]:
def token_logprobs_from_logits(logits: torch.Tensor, input_ids: torch.Tensor) -> torch.Tensor:
    """Return log p(token_t | token_<t) with shape [batch, time-1]."""
    if logits.ndim != 3 or input_ids.ndim != 2 or logits.shape[:2] != input_ids.shape:
        raise ValueError("logits [B,T,V] and input_ids [B,T] must align")
    log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
    return log_probs.gather(-1, input_ids[:, 1:].unsqueeze(-1)).squeeze(-1)


def sequence_logprob(logits: torch.Tensor, input_ids: torch.Tensor, response_mask: torch.Tensor, length_normalize: bool = False) -> torch.Tensor:
    token_logprobs = token_logprobs_from_logits(logits, input_ids)
    mask = response_mask[:, 1:].to(token_logprobs.dtype)
    values = (token_logprobs * mask).sum(dim=-1)
    if length_normalize:
        values = values / mask.sum(dim=-1).clamp_min(1.0)
    return values


def dpo_loss(
    policy_chosen_logp: torch.Tensor,
    policy_rejected_logp: torch.Tensor,
    reference_chosen_logp: torch.Tensor,
    reference_rejected_logp: torch.Tensor,
    beta: float = DPO_BETA,
) -> dict[str, torch.Tensor]:
    chosen_advantage = policy_chosen_logp - reference_chosen_logp
    rejected_advantage = policy_rejected_logp - reference_rejected_logp
    logits = beta * (chosen_advantage - rejected_advantage)
    loss = -F.logsigmoid(logits).mean()
    return {
        "loss": loss,
        "preference_logits": logits.detach(),
        "chosen_reward_margin": (beta * chosen_advantage).detach(),
        "rejected_reward_margin": (beta * rejected_advantage).detach(),
    }


def dpo_step(
    policy: nn.Module,
    reference: nn.Module,
    batch: dict[str, torch.Tensor],
    optimizer: torch.optim.Optimizer,
    beta: float = DPO_BETA,
) -> float:
    policy.train()
    reference.eval()
    policy_chosen = policy(batch["chosen_input_ids"])["logits"]
    policy_rejected = policy(batch["rejected_input_ids"])["logits"]
    with torch.no_grad():
        reference_chosen = reference(batch["chosen_input_ids"])["logits"]
        reference_rejected = reference(batch["rejected_input_ids"])["logits"]
    values = dpo_loss(
        sequence_logprob(policy_chosen, batch["chosen_input_ids"], batch["chosen_response_mask"]),
        sequence_logprob(policy_rejected, batch["rejected_input_ids"], batch["rejected_response_mask"]),
        sequence_logprob(reference_chosen, batch["chosen_input_ids"], batch["chosen_response_mask"]),
        sequence_logprob(reference_rejected, batch["rejected_input_ids"], batch["rejected_response_mask"]),
        beta,
    )
    optimizer.zero_grad(set_to_none=True)
    values["loss"].backward()
    optimizer.step()
    return float(values["loss"].detach())

## 6. GPU Rollout 和 DPO 训练入口（当前关闭）

真实运行时先激活独立 vLLM Profile 生成每题 4 条候选，保存实际 Completion Token IDs / Logprobs / 截断状态，再关闭 vLLM 切换训练 Profile。下面的入口明确区分 Rollout 和 DPO 更新；CPU 阶段不会触发任何导入或模型分配。

### 6.1 读取并校验 SFT Manifest

In [ ]:
def read_sft_manifest(path: Path = SFT_MANIFEST_PATH) -> dict[str, object]:
    if not path.exists():
        raise FileNotFoundError(f"SFT Manifest 不存在：{path}；请先运行 Notebook 03")
    manifest = json.loads(path.read_text(encoding="utf-8"))
    if manifest.get("schema_version") != "nemotron-sft-checkpoint/v1":
        raise ValueError("Unsupported SFT manifest schema")
    if manifest.get("base_model", {}).get("id") != MODEL_ID:
        raise ValueError("SFT Manifest 的 Base Model 与课程固定模型不一致")
    if manifest.get("status") != "completed":
        raise ValueError("SFT Manifest 尚未标记为 completed，不能初始化 DPO")
    adapter_path = Path(resolve_manifest_path(str(manifest.get("adapter_path", ""))))
    if not adapter_path.is_dir() or not (adapter_path / "adapter_config.json").exists():
        raise FileNotFoundError(f"SFT Adapter 不存在或缺少 adapter_config.json：{adapter_path}")
    return manifest


def resolve_manifest_path(value: str) -> str:
    path = Path(value)
    return str(path if path.is_absolute() else PROJECT_ROOT / path)


### 6.2 解析 vLLM 行为输出的 Token Logprob

In [ ]:
def _chosen_logprob_values(generated) -> list[float]:
    raw, token_ids = generated.logprobs, list(generated.token_ids or [])
    if raw is None or len(raw) != len(token_ids):
        raise RuntimeError("vLLM logprobs 与 Completion Token IDs 不一致")
    values = []
    for token_id, candidates in zip(token_ids, raw):
        selected = candidates.get(token_id, candidates.get(str(token_id))) if isinstance(candidates, dict) else candidates
        if selected is None and isinstance(candidates, dict) and len(candidates) == 1:
            selected = next(iter(candidates.values()))
        value = getattr(selected, "logprob", selected)
        if value is None or not math.isfinite(float(value)):
            raise RuntimeError("vLLM 返回了缺失或非有限行为策略 Logprob")
        values.append(float(value))
    return values


### 6.3 vLLM Rollout：一次生成多条候选

In [ ]:
def run_vllm_rollout(prompts: list[dict[str, object]], model_source: str | None, policy_version: str, adapter_path: str | None = None) -> list[dict[str, object]]:
    """Use the vLLM Profile for rollout; adapter is optional but explicit."""
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    model_source = model_source or BASE_MODEL_PATH
    tokenizer = AutoTokenizer.from_pretrained(model_source, trust_remote_code=True, local_files_only=Path(model_source).exists())
    llm_kwargs = {"model": model_source, "tokenizer": model_source, "tensor_parallel_size": 1, "gpu_memory_utilization": 0.85, "max_model_len": 8192, "enforce_eager": True}
    lora_request = None
    if adapter_path:
        from vllm.lora.request import LoRARequest
        llm_kwargs["enable_lora"] = True
        lora_request = LoRARequest("sft", 1, adapter_path)
    llm = LLM(**llm_kwargs)
    sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_NEW_TOKENS, n=NUM_CANDIDATES_PER_PROMPT, logprobs=1)
    rendered = [tokenizer.apply_chat_template([{"role": "user", "content": row["prompt"]}], tokenize=False, add_generation_prompt=True, enable_thinking=True) for row in prompts]
    outputs = llm.generate(rendered, sampling, use_tqdm=True, lora_request=lora_request)
    records = []
    for row, request in zip(prompts, outputs):
        prompt_token_ids = list(getattr(request, "prompt_token_ids", []) or [])
        if len(request.outputs) != NUM_CANDIDATES_PER_PROMPT:
            raise RuntimeError("vLLM 未返回要求数量的 DPO 候选")
        for generated in request.outputs:
            completion_ids = list(generated.token_ids or [])
            truncated = bool(generated.finish_reason == "length")
            correct, method = verify_completion(str(row["task_type"]), generated.text, str(row["answer"]), truncated=truncated)
            records.append(record_to_dict(RolloutRecord(
                sample_id=str(row["id"]), task_type=str(row["task_type"]), prompt=str(row["prompt"]), completion=generated.text, expected_answer=str(row["answer"]), completion_token_ids=completion_ids, prompt_token_ids=prompt_token_ids, response_mask=[0] * len(prompt_token_ids) + [1] * len(completion_ids), finish_reason=str(generated.finish_reason), truncated=truncated, policy_version=policy_version, verifier_version=OFFICIAL_VERIFIER_VERSION, temperature=TEMPERATURE, top_p=TOP_P, source="vllm_rollout", fixture=False, correct=correct, verification_method=method)))
    del llm
    try:
        import gc; gc.collect()
        import torch
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except Exception: pass
    return records


### 6.4 Chosen/Rejected 的 Token 化与 Padding

In [ ]:
def _encode_preference_pair(tokenizer, prompt: str, completion: str, max_length: int = 2048) -> dict[str, list[int]]:
    messages, assistant = [{"role": "user", "content": prompt}], {"role": "assistant", "content": completion}
    try:
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
        full_text = tokenizer.apply_chat_template(messages + [assistant], tokenize=False, add_generation_prompt=False, enable_thinking=True)
    except TypeError:
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        full_text = tokenizer.apply_chat_template(messages + [assistant], tokenize=False, add_generation_prompt=False)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
    full_ids = tokenizer(full_text, add_special_tokens=False).input_ids
    if len(full_ids) < len(prompt_ids) or full_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError("preference Chat Template 的 Prompt 前缀不一致")
    full_ids = full_ids[:max_length]
    if len(full_ids) <= len(prompt_ids):
        raise ValueError("preference completion 被截断为空")
    return {"input_ids": full_ids, "response_mask": [0] * len(prompt_ids) + [1] * (len(full_ids) - len(prompt_ids))}


def _collate_preference_examples(examples: list[dict[str, list[int]]], pad_token_id: int) -> dict[str, torch.Tensor]:
    max_len = max(max(len(example["chosen_input_ids"]), len(example["rejected_input_ids"])) for example in examples)
    values = {"chosen_input_ids": [], "rejected_input_ids": [], "chosen_response_mask": [], "rejected_response_mask": []}
    for example in examples:
        for prefix in ("chosen", "rejected"):
            ids, mask = example[f"{prefix}_input_ids"], example[f"{prefix}_response_mask"]
            pad = max_len - len(ids)
            values[f"{prefix}_input_ids"].append(ids + [pad_token_id] * pad)
            values[f"{prefix}_response_mask"].append(mask + [0] * pad)
    return {key: torch.tensor(value, dtype=torch.long) for key, value in values.items()}


### 6.5 从同一 SFT Adapter 初始化 Policy 与冻结 Reference

In [ ]:
def _load_sft_policy_and_reference(manifest: dict[str, object], mode: str):
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel, prepare_model_for_kbit_training
    base_path = manifest["base_model"]["path"]
    adapter_path = resolve_manifest_path(str(manifest["adapter_path"]))
    tokenizer_path = manifest.get("tokenizer_path", base_path)
    kwargs = {"trust_remote_code": True, "device_map": "auto"}
    if mode == "nf4_qlora":
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    elif mode == "bf16_lora":
        kwargs["torch_dtype"] = torch.bfloat16
    else:
        raise ValueError(mode)
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, trust_remote_code=True)
    base_policy = AutoModelForCausalLM.from_pretrained(base_path, **kwargs)
    base_reference = AutoModelForCausalLM.from_pretrained(base_path, **kwargs)
    if mode == "nf4_qlora":
        base_policy = prepare_model_for_kbit_training(base_policy, use_gradient_checkpointing=True)
        base_reference = prepare_model_for_kbit_training(base_reference, use_gradient_checkpointing=True)
    policy = PeftModel.from_pretrained(base_policy, adapter_path, is_trainable=True)
    reference = PeftModel.from_pretrained(base_reference, adapter_path, is_trainable=False)
    for parameter in reference.parameters(): parameter.requires_grad_(False)
    return policy, reference, tokenizer, adapter_path


### 6.6 Adapter 产物可重载性检查

In [ ]:
def validate_saved_adapter_artifact(adapter_path: Path) -> dict[str, object]:
    config = adapter_path / "adapter_config.json"
    weights = list(adapter_path.glob("adapter_model.*"))
    return {"path": str(adapter_path), "config_exists": config.exists(), "weights_exists": bool(weights), "reloadable_artifact": config.exists() and bool(weights)}


### 6.7 DPO Update 与 Adapter 保存

In [ ]:
def run_dpo_training(preference_pairs: list[dict[str, object]], sft_manifest_path: Path = SFT_MANIFEST_PATH, mode: str = "bf16_lora") -> dict[str, object]:
    if not RUN_DPO:
        return {"status": "skipped_by_default", "reason": "RUN_DPO=False；DGX Spark 阶段不加载模型。"}
    if not preference_pairs:
        return {"status": "blocked_no_preference_pairs", "reason": "没有经过 Verifier 的 Chosen/Rejected 对"}
    manifest = read_sft_manifest(sft_manifest_path)
    policy, reference, tokenizer, parent_adapter = _load_sft_policy_and_reference(manifest, mode)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    encoded_pairs = []
    for pair in preference_pairs:
        chosen = _encode_preference_pair(tokenizer, str(pair["prompt"]), str(pair["chosen"]))
        rejected = _encode_preference_pair(tokenizer, str(pair["prompt"]), str(pair["rejected"]))
        encoded_pairs.append({"chosen_input_ids": chosen["input_ids"], "chosen_response_mask": chosen["response_mask"], "rejected_input_ids": rejected["input_ids"], "rejected_response_mask": rejected["response_mask"]})
    optimizer = torch.optim.AdamW((parameter for parameter in policy.parameters() if parameter.requires_grad), lr=LEARNING_RATE)
    history, started = [], time.perf_counter()
    for epoch in range(1):
        losses = []
        for start in range(0, len(encoded_pairs), 1):
            batch = _collate_preference_examples(encoded_pairs[start:start + 1], pad_id)
            device = next(policy.parameters()).device
            batch = {key: value.to(device) for key, value in batch.items()}
            losses.append(dpo_step(policy, reference, batch, optimizer, DPO_BETA))
        history.append({"epoch": epoch + 1, "loss": float(np.mean(losses)) if losses else float("nan")})
    output_dir = CHECKPOINT_ROOT / f"dpo_{mode}"
    output_dir.mkdir(parents=True, exist_ok=True)
    policy.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    artifact = validate_saved_adapter_artifact(output_dir)
    dpo_manifest = {"schema_version": "nemotron-dpo-checkpoint/v1", "status": "completed", "parent_sft_manifest": str(sft_manifest_path.relative_to(PROJECT_ROOT)), "parent_sft_adapter": str(parent_adapter), "base_model": manifest["base_model"], "tokenizer_path": manifest.get("tokenizer_path"), "adapter_path": str(output_dir.relative_to(PROJECT_ROOT)), "mode": mode, "pairs": len(preference_pairs), "seed": SEED, "policy_version": f"dpo:{mode}:seed{SEED}", "profile": TRAINING_PROFILE}
    (output_dir / "dpo_manifest.json").write_text(json.dumps(dpo_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    return {"status": "completed", "pairs": len(preference_pairs), "history": history, "training_time_seconds": time.perf_counter() - started, "reference_policy": "frozen same SFT Adapter", "adapter_path": str(output_dir), "adapter_reload_check": artifact}


### 6.8 DPO Smoke Pipeline：Rollout → Verifier → Preference → Update

In [ ]:
def run_dpo_smoke_pipeline(mode: str = "nf4_qlora") -> dict[str, object]:
    """SFT Adapter -> vLLM Rollout -> Verifier -> preference pairs -> one DPO update."""
    if not (RUN_ROLLOUT or RUN_DPO):
        return {"status": "skipped_by_default", "reason": "RUN_ROLLOUT/RUN_DPO 均为 False"}
    manifest = read_sft_manifest()
    adapter_path = resolve_manifest_path(str(manifest["adapter_path"]))
    prompts = select_smoke_prompts()
    rollouts = run_vllm_rollout(prompts, manifest["base_model"]["path"], str(manifest.get("policy_version", "sft_policy")), adapter_path=adapter_path)
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    rollout_path = RESULT_DIR / "dpo_smoke_rollouts.jsonl"
    rollout_path.write_text("\n".join(json.dumps(row, ensure_ascii=False) for row in rollouts) + "\n", encoding="utf-8")
    pairs = construct_preference_pairs(rollouts, "dpo_a")
    pair_audit = audit_preference_pairs(pairs)
    if pair_audit["valid_pairs"] == 0:
        return {"status": "blocked_no_preference_pairs", "rollout_path": str(rollout_path), "preference_audit": pair_audit}
    training = run_dpo_training(pairs, SFT_MANIFEST_PATH, mode)
    return {"status": "completed", "rollout_path": str(rollout_path), "preference_audit": pair_audit, "training": training}


### 6.9 GPU Rollout / DPO 开关

In [ ]:
if RUN_ROLLOUT or RUN_DPO:
    print({"gpu_entry": "enabled_when_explicitly_requested", "training_profile": TRAINING_PROFILE})
else:
    print({"rollout": "disabled_by_default", "dpo_training": "disabled_by_default", "reference_policy": "same_sft_adapter"})


## 7. CPU Smoke Test：偏好质量、Log Probability 和梯度

用两个小型语言模型和手工 Token 序列验证：Chosen/Rejected 标签正确、Reference 不产生梯度、DPO Loss 有限且当前 Policy 获得梯度。测试通过不代表真实 DPO 已运行。

In [ ]:
class TinyLM(nn.Module):
    def __init__(self, vocab_size: int = 24, hidden_size: int = 12):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.head = nn.Linear(hidden_size, vocab_size)
    def forward(self, input_ids):
        return {"logits": self.head(self.embedding(input_ids))}


def make_toy_preference_batch(batch_size: int = 2, time_steps: int = 6, vocab_size: int = 24) -> dict[str, torch.Tensor]:
    torch.manual_seed(SEED)
    chosen = torch.randint(0, vocab_size, (batch_size, time_steps))
    rejected = torch.randint(0, vocab_size, (batch_size, time_steps))
    mask = torch.zeros(batch_size, time_steps, dtype=torch.long)
    mask[:, 2:] = 1
    return {
        "chosen_input_ids": chosen,
        "rejected_input_ids": rejected,
        "chosen_response_mask": mask,
        "rejected_response_mask": mask.clone(),
    }


policy = TinyLM()
reference = TinyLM()
reference.load_state_dict(policy.state_dict())
for parameter in reference.parameters():
    parameter.requires_grad_(False)
optimizer = torch.optim.AdamW(policy.parameters(), lr=1e-2)
batch = make_toy_preference_batch()
old_reference = [parameter.detach().clone() for parameter in reference.parameters()]
loss_value = dpo_step(policy, reference, batch, optimizer)
assert math.isfinite(loss_value)
assert all(parameter.grad is None for parameter in reference.parameters())
assert any(parameter.grad is not None for parameter in policy.parameters())
assert all(torch.equal(before, after) for before, after in zip(old_reference, reference.parameters()))
assert audit_preference_pairs(preference_pairs)["valid_pairs"] == len(preference_pairs)
# Explicit final markers win over unrelated numbers in the Thinking trace.
assert verify_completion("gravity", "Step 1 uses t=2.0; Step 2 uses 7.0.\n\\boxed{42.0}", "42.0")[0]
assert verify_completion("gravity", "Step 1 uses 2.0; final token budget is 99", "42.0")[0] is False
assert verify_completion("gravity", "\\boxed{42.0}", "42.0", truncated=True)[1] == "truncated_output"
# Standard DPO keeps sequence log-probability as a sum, so a longer response
# contributes more token log-probability without length normalization.
zero_short = torch.zeros(1, 3, 24)
zero_long = torch.zeros(1, 4, 24)
short_ids = torch.tensor([[1, 2, 3]])
long_ids = torch.tensor([[1, 2, 3, 4]])
short_mask = torch.tensor([[0, 1, 1]])
long_mask = torch.tensor([[0, 1, 1, 1]])
short_logp = sequence_logprob(zero_short, short_ids, short_mask)
long_logp = sequence_logprob(zero_long, long_ids, long_mask)
assert long_logp.item() < short_logp.item()

dpo_cpu_report = {
    "status": "passed",
    "fixture_rollouts": len(fixture_rollouts),
    "fixture_preference_pairs": len(preference_pairs),
    "preference_audit": audit_preference_pairs(preference_pairs),
    "readiness_audit": rl_readiness_audit(fixture_rollouts),
    "dpo_loss_finite": True,
    "reference_frozen": True,
    "reference_initialization_match": True,
    "verifier_version": OFFICIAL_VERIFIER_VERSION,
    "policy_gradient": True,
    "gpu_rollout": "not_run",
    "gpu_dpo": "not_run",
}
print(json.dumps(dpo_cpu_report, ensure_ascii=False, indent=2))

## 本节总结与限制

CPU 阶段完成了偏好数据结构、Verifier 标签、RL Readiness Audit、Response Log Probability、Reference Freeze 和 DPO 梯度 Smoke Test。真实 Rollout、Adapter 训练、保存/重载和 SFT→DPO 指标仍是 `Not Run`，不能把 Fixture 对作为实验结果。

明天必须先选定并固定最佳 SFT Checkpoint，再在 RTX 5090 上对 Equation Numeric / Bit Manipulation 生成少量真实候选，统计偏好对产生率和标签质量，最后运行小步 DPO。若偏好对过少，应记录为数据覆盖问题，而不是强行训练。

## 8. SFT Manifest 交接检查

真实 DPO 更新只在 `RUN_DPO=True` 时执行；CPU 阶段检查入口会读取同一 SFT Manifest，Adapter 目录不再直接传给 `AutoModelForCausalLM`。


In [ ]:
if SFT_MANIFEST_PATH.exists():
    dpo_manifest_probe = read_sft_manifest(SFT_MANIFEST_PATH)
    assert dpo_manifest_probe["base_model"]["id"] == MODEL_ID
    dpo_manifest_status = "available"
else:
    dpo_manifest_status = "not_run_sft_manifest_missing"
assert RUN_DPO is False and RUN_ROLLOUT is False
dpo_checkpoint_report = {"status":"passed", "smoke_prompts": len(select_smoke_prompts()), "candidates_per_prompt": NUM_CANDIDATES_PER_PROMPT, "preference_pairs": len(preference_pairs), "reference_frozen": True, "reference_initialization_match": True, "sequence_logprob_is_sum": True, "verifier_version": OFFICIAL_VERIFIER_VERSION, "sft_manifest": dpo_manifest_status, "gpu": "not_run"}
(RESULT_DIR / "cpu_dpo_smoke_report.json").parent.mkdir(parents=True, exist_ok=True)
(RESULT_DIR / "cpu_dpo_smoke_report.json").write_text(json.dumps(dpo_checkpoint_report, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(dpo_checkpoint_report, ensure_ascii=False, indent=2))
